In [11]:
from neural_networks.Recurrent import RNN, FusionNN
from dataLoader import *
import numpy as np
import torch

In [12]:
import numpy as np
import torch
from sklearn.decomposition import PCA

def resize_tensor(x, size=(7, 7)):
    x = x.unsqueeze(0)
    x = torch.nn.functional.interpolate(x, size=size, mode='bilinear', align_corners=False)
    return x.squeeze(0)

def load_images(dirName, reduce=1):
    files_train = sorted(dirName.glob("*.npz"))
    train_data = []

    for _, file_name in enumerate(files_train):
        with np.load(file_name) as npz:
            arr = np.ma.MaskedArray(**npz)
            data = arr.data
            data = np.append(data, [arr.mask[0]], axis=0)
            
            x = torch.tensor(data, dtype=torch.float32)
            x = resize_tensor(x)  # Shape: (C, 7, 7)
            train_data.append(x)

    if not train_data:
        return []

    max_width = max(data.shape[1] for data in train_data)
    max_height = max(data.shape[2] for data in train_data)
    max_size = max_width if max_width > max_height else max_height

    if reduce == 1:
        return [resize_tensor(data) for data in train_data]
        
    dataset_tensor = torch.stack(train_data)
    N, C, H, W = dataset_tensor.shape

    pixels = dataset_tensor.permute(0, 2, 3, 1).reshape(-1, C).numpy()

    pca = PCA(n_components=reduce)
    reduced_pixels = pca.fit_transform(pixels)

    actual_components = pca.n_components_

    reduced_tensor = torch.tensor(reduced_pixels, dtype=torch.float32)
    reduced_tensor = reduced_tensor.view(N, H, W, actual_components).permute(0, 3, 1, 2)

    return ([img for img in reduced_tensor], pca)

train_data = load_images(HSI_SATELLITE_DIR)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

msi = load_images(MSI_SATELLITE_TRAIN_DIR)
hsi = load_images(HSI_SATELLITE_DIR)
airborne = load_images(HSI_AIRBORNE_DIR)
y = gt_train_df[column_names].values
y = scaler.fit_transform(y)
y_binned = np.digitize(y, bins=np.linspace(y.min(), y.max() + 1, 5))
y = torch.tensor(y, dtype=torch.float32)
device = "cuda" if torch.cuda.is_available else "cpu"

In [14]:
from torch.utils.data import Dataset

class MultiSourceDataset(Dataset):
    def __init__(self, msi, hsi, airborne, targets):
        self.msi = msi
        self.hsi = hsi
        self.airborne = airborne
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.msi[idx],
            self.hsi[idx],
            self.airborne[idx],
            self.targets[idx]
        )

In [15]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

dataset = MultiSourceDataset(msi, hsi, airborne, y)

indices = list(range(len(dataset)))
train_idx, val_idx = train_test_split(indices, train_size=0.8)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

#Dimensionality reduction

model = RNN(neurons=256, hidden_size=2048).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.MSELoss()

early_stopping_treshold = 64
best_val_loss = float('inf')

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [9]:
early_stopping_counter = 0
# model.load_state_dict(torch.load("models/nn/RNN.pt"))
# early_stopping_before = best_val_loss
for epoch in range(1000):
    model.train()
    train_loss = 0.0

    for msi_x, hsi_x, air_x, y in train_loader:
        msi_x = msi_x.to(device)
        hsi_x = hsi_x.to(device)
        air_x = air_x.to(device)
        y = y.to(device)

        output = model([msi_x, hsi_x, air_x])
        loss = loss_fn(output, y)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for msi_x, hsi_x, air_x, y in val_loader:
            msi_x = msi_x.to(device)
            hsi_x = hsi_x.to(device)
            air_x = air_x.to(device)
            y = y.to(device)

            output = model([msi_x, hsi_x])

            loss = loss_fn(output, y)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "models/nn/RNN_95.pt")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= early_stopping_treshold:
            break

print(best_val_loss)

Epoch: 0, train loss: 0.6878 val loss: 0.6595
Epoch: 1, train loss: 0.5931 val loss: 0.5938
Epoch: 2, train loss: 0.5701 val loss: 0.5579
Epoch: 3, train loss: 0.5555 val loss: 0.5347
Epoch: 4, train loss: 0.5376 val loss: 0.5551
Epoch: 5, train loss: 0.5250 val loss: 0.5178
Epoch: 6, train loss: 0.5175 val loss: 0.5249
Epoch: 7, train loss: 0.5042 val loss: 0.5230
Epoch: 8, train loss: 0.4996 val loss: 0.5257
Epoch: 9, train loss: 0.4955 val loss: 0.5526
Epoch: 10, train loss: 0.5038 val loss: 0.5146
Epoch: 11, train loss: 0.4816 val loss: 0.5608
Epoch: 12, train loss: 0.4957 val loss: 0.5184
Epoch: 13, train loss: 0.4857 val loss: 0.5189
Epoch: 14, train loss: 0.4873 val loss: 0.5264
Epoch: 15, train loss: 0.5021 val loss: 0.5405
Epoch: 16, train loss: 0.5207 val loss: 0.5229
Epoch: 17, train loss: 0.4833 val loss: 0.5067
Epoch: 18, train loss: 0.4801 val loss: 0.5049
Epoch: 19, train loss: 0.4695 val loss: 0.5204
Epoch: 20, train loss: 0.4692 val loss: 0.5297
Epoch: 21, train loss: 

In [16]:
from BaselineRegressor import BaselineRegressor
from sklearn.metrics import mean_squared_error

X_baseline = load_msi_data(HSI_SATELLITE_DIR)
y_baseline = gt_train_df[column_names].values
baseline = BaselineRegressor()
baseline.fit(X_baseline, y_baseline)
y_baseline_pred = baseline.predict(X_baseline)
mse_baseline = mean_squared_error(y_baseline_pred, y_baseline, multioutput='raw_values')

model.load_state_dict(torch.load("models/nn/FusionNN.pt"))
model.eval()
mse_loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
predictions = []
targets = []

with torch.no_grad():
    for msi_x, hsi_x, air_x, y in mse_loader:

        # hsi_x: (B, C, H, W)
        B, C, H, W = hsi_x.shape

        # Convert each pixel into one sample
        pixels = (
            hsi_x.permute(0, 2, 3, 1)
                 .reshape(-1, C)
                 .cpu()
                 .numpy()
        )

        # Apply PCA
        # reduced_pixels = pca.transform(pixels)

        # Convert back to (B, C_new, H, W)
        # hsi_x = (
        #     torch.tensor(reduced_pixels, dtype=torch.float32)
        #     .view(B, H, W, pca.n_components_)
        #     .permute(0, 3, 1, 2)
        # )

        msi_x = msi_x.to(device)
        hsi_x = hsi_x.to(device)
        air_x = air_x.to(device)
        y = y.to(device)

        output = model([msi_x, hsi_x])
        predictions.append(output.cpu())
        targets.append(y.cpu())

    predictions = torch.cat(predictions, dim=0).numpy()
    predictions = scaler.inverse_transform(predictions)
    targets = torch.cat(targets, dim=0).numpy()

mse = mean_squared_error(predictions, y_baseline, multioutput='raw_values')
score = mse / mse_baseline
score

array([0.39059347, 0.41592714, 0.30646235, 0.49188126, 0.50427348,
       0.40915999])

In [8]:
from pandas import DataFrame

predictions_score = model([msi, hsi])

files_test_msi = sorted(MSI_SATELLITE_TEST_DIR.glob("*.npz"))
test_data_msi = []
for file in enumerate(files_test_msi):
    file_name = file[1]
    with np.load(file_name) as npz:
        arr = np.ma.MaskedArray(**npz)
        data = arr.data
        data = np.append(data, [arr.mask[0]], axis=0)
        x = torch.tensor(data, dtype=torch.float32)
        x = resize_tensor(x)
        test_data_msi.append(x)
        
max_width = max(data.shape[1] for data in test_data_msi)
max_height = max(data.shape[2] for data in test_data_msi)
max_size = max_width if max_width > max_height else max_height

test_data_msi = [resize_tensor(data) for data in test_data_msi]
test_data_msi = torch.stack(test_data_msi).to(device)

files_test_hsi = sorted(HSI_SATELLITE_TEST_DIR.glob("*.npz"))
test_data_hsi = []
for file in enumerate(files_test_hsi):
    file_name = file[1]
    with np.load(file_name) as npz:
        arr = np.ma.MaskedArray(**npz)
        data = arr.data
        data = np.append(data, [arr.mask[0]], axis=0)
        x = torch.tensor(data, dtype=torch.float32)
        x = resize_tensor(x)

        #pca
        C, H, W = x.shape
        pixels = (
            x.permute(1, 2, 0)   # (7, 7, 231)
             .reshape(-1, C)     # (49, 231)
             .numpy()
        )
        pixels_reduced = pca.transform(pixels)  # (49, n_components)
        x = (
            torch.tensor(pixels_reduced, dtype=torch.float32)
            .view(H, W, pca.n_components_)
            .permute(2, 0, 1)    # (n_components, 7, 7)
        )
        # end of PCA
        test_data_hsi.append(x)
        
max_width = max(data.shape[1] for data in test_data_msi)
max_height = max(data.shape[2] for data in test_data_msi)
max_size = max_width if max_width > max_height else max_height

test_data_hsi = [resize_tensor(data) for data in test_data_hsi]
test_data_hsi = torch.stack(test_data_hsi).to(device)
with torch.no_grad():
    predictions = model([test_data_msi, test_data_hsi])

predictions = predictions.cpu().numpy()
predictions = scaler.inverse_transform(predictions)

submission = DataFrame(data = predictions, columns = column_names)
submission.to_csv("recurrent_95.csv", index_label="sample_index")

# Alternative: Fusing CNNs to neural network without recurrent NN

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True) # any change
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

# 8192 12000
model = FusionNN(neurons=256, hidden_size=12000).to(device) #only increase

criterion = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-6) # Fine
loss_fn = torch.nn.MSELoss()

# 42580369114875793
early_stopping_treshold = 64
best_val_loss = float('inf')

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [ ]:
model.load_state_dict(torch.load("models/nn/FusionNN.pt"))

In [12]:
early_stopping_before = best_val_loss
early_stopping_counter = 0

for epoch in range(5000):
    model.train()
    train_loss = 0.0

    for msi_x, hsi_x, air_x, y in train_loader:
        msi_x = msi_x.to(device)
        hsi_x = hsi_x.to(device)
        air_x = air_x.to(device)
        y = y.to(device)

        output = model([msi_x, hsi_x])
        loss = loss_fn(output, y)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for msi_x, hsi_x, air_x, y in val_loader:
    
            # hsi_x: (B, C, H, W)
            B, C, H, W = hsi_x.shape
    
            # Convert each pixel into one sample
            pixels = (
                hsi_x.permute(0, 2, 3, 1)
                     .reshape(-1, C)
                     .cpu()
                     .numpy()
            )
    
            # Apply PCA
            # reduced_pixels = pca.transform(pixels)
    
            # Convert back to (B, C_new, H, W)
            # hsi_x = (
            #     torch.tensor(reduced_pixels, dtype=torch.float32)
            #     .view(B, H, W, pca.n_components_)
            #     .permute(0, 3, 1, 2)
            # )
    
            msi_x = msi_x.to(device)
            hsi_x = hsi_x.to(device)
            air_x = air_x.to(device)
            y = y.to(device)
    
            output = model([msi_x, hsi_x])
    
            loss = loss_fn(output, y)
            val_loss += loss.item()


    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "models/nn/FusionNN_95.pt")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= early_stopping_treshold:
            print(epoch, best_val_loss)
            break

Epoch: 0, train loss: 0.7457 val loss: 0.8057
Epoch: 1, train loss: 0.6954 val loss: 0.7637
Epoch: 2, train loss: 0.6633 val loss: 0.7341
Epoch: 3, train loss: 0.6330 val loss: 0.7085
Epoch: 4, train loss: 0.6066 val loss: 0.6859
Epoch: 5, train loss: 0.5872 val loss: 0.6657
Epoch: 6, train loss: 0.5706 val loss: 0.6463
Epoch: 7, train loss: 0.5542 val loss: 0.6302
Epoch: 8, train loss: 0.5427 val loss: 0.6182
Epoch: 9, train loss: 0.5294 val loss: 0.6048
Epoch: 10, train loss: 0.5177 val loss: 0.5930
Epoch: 11, train loss: 0.5117 val loss: 0.5881
Epoch: 12, train loss: 0.5035 val loss: 0.5832
Epoch: 13, train loss: 0.5012 val loss: 0.5777
Epoch: 14, train loss: 0.4964 val loss: 0.5756
Epoch: 15, train loss: 0.4918 val loss: 0.5648
Epoch: 16, train loss: 0.4864 val loss: 0.5623
Epoch: 17, train loss: 0.4875 val loss: 0.5630
Epoch: 18, train loss: 0.4829 val loss: 0.5569
Epoch: 19, train loss: 0.4816 val loss: 0.5598
Epoch: 20, train loss: 0.4763 val loss: 0.5565
Epoch: 21, train loss: 

# Alternative 2: RNN with bands as timestamps instead of sources of images

In [28]:
from torch.utils.data import Dataset

class BandDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, idx):
        return (self.x[idx], self.y[idx])

In [35]:
from neural_networks.Recurrent import BandRNN
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch.utils.data import Subset

scaler = StandardScaler()
hsi, pca = load_images(HSI_SATELLITE_DIR, 0.95)
y = gt_train_df[column_names].values
y = scaler.fit_transform(y)
y_binned = np.digitize(y, bins=np.linspace(y.min(), y.max() + 1, 5))
y = torch.tensor(y, dtype=torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"

indices = list(range(len(hsi)))
dataset = BandDataset(hsi,y)
indices = list(range(len(dataset)))

train_idx, val_idx = train_test_split(
    indices,
    train_size=0.8,
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

from torch.utils.data import Subset

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

model = BandRNN(rnn_hidden_size=128, hidden_size=4096, in_bands=3).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.MSELoss()

early_stopping_treshold = 64
best_val_loss = float('inf')

In [12]:
early_stopping_counter = 0
for epoch in range(5000):
    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = loss_fn(output, y)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)

            loss = loss_fn(output, y)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "models/nn/bands_hsi_95.pt")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= early_stopping_treshold:
            break

print(best_val_loss)

Epoch: 0, train loss: 0.9627 val loss: 0.9021
Epoch: 1, train loss: 0.8758 val loss: 0.8784
Epoch: 2, train loss: 0.8491 val loss: 0.8532
Epoch: 3, train loss: 0.8316 val loss: 0.8353
Epoch: 4, train loss: 0.8113 val loss: 0.8197
Epoch: 5, train loss: 0.8178 val loss: 0.8270
Epoch: 6, train loss: 0.8013 val loss: 0.8137
Epoch: 7, train loss: 0.7932 val loss: 0.8253
Epoch: 8, train loss: 0.7844 val loss: 0.8132
Epoch: 9, train loss: 0.7714 val loss: 0.7866
Epoch: 10, train loss: 0.7670 val loss: 0.7841
Epoch: 11, train loss: 0.7657 val loss: 0.7952
Epoch: 12, train loss: 0.7594 val loss: 0.7865
Epoch: 13, train loss: 0.7581 val loss: 0.7883
Epoch: 14, train loss: 0.7521 val loss: 0.7782
Epoch: 15, train loss: 0.7543 val loss: 0.7540
Epoch: 16, train loss: 0.7458 val loss: 0.7596
Epoch: 17, train loss: 0.7469 val loss: 0.7542
Epoch: 18, train loss: 0.7519 val loss: 0.7623
Epoch: 19, train loss: 0.7459 val loss: 0.7714
Epoch: 20, train loss: 0.7435 val loss: 0.7561
Epoch: 21, train loss: 

In [13]:
from pandas import DataFrame

files_test = sorted(HSI_SATELLITE_TEST_DIR.glob("*.npz"))
test_data = []
for file in enumerate(files_test):
    file_name = file[1]
    with np.load(file_name) as npz:
        arr = np.ma.MaskedArray(**npz)
        data = arr.data
        data = np.append(data, [arr.mask[0]], axis=0)
        x = torch.tensor(data, dtype=torch.float32)
        x = resize_tensor(x)
        test_data.append(x)
        
max_width = max(data.shape[1] for data in test_data)
max_height = max(data.shape[2] for data in test_data)
max_size = max_width if max_width > max_height else max_height

# test_data = torch.stack(test_data).to(device)
# with torch.no_grad():
#     predictions = model(test_data)

# For PCA
test_tensor = torch.stack(test_data)      # (N, C, H, W)
N, C, H, W = test_tensor.shape

pixels = test_tensor.permute(0, 2, 3, 1).reshape(-1, C).numpy()

reduced_pixels = pca.transform(pixels)

actual_components = pca.n_components_

test_tensor = (
    torch.tensor(reduced_pixels, dtype=torch.float32)
    .view(N, H, W, actual_components)
    .permute(0, 3, 1, 2)
)

test_tensor = test_tensor.to(device)

with torch.no_grad():
    predictions = model(test_tensor)
# End of PCA
    
predictions = predictions.cpu().numpy()
predictions = scaler.inverse_transform(predictions)

submission = DataFrame(data = predictions, columns = column_names)
submission.to_csv("bandHSI_95.csv", index_label="sample_index")

RuntimeError: Given groups=1, weight of size [256, 13, 3, 3], expected input[1, 3, 7, 7] to have 13 channels, but got 3 channels instead